# MeteoPrep — exploration

Notebook d'exploration de l'étape 2 : météo mensuelle par hôtel.

**Principe :** les champs d'identification (`hotel_code` … `hotel_lat`, `hotel_lon`) viennent de RodPrep.
La météo est calculée au point `(hotel_lat, hotel_lon)` (stations Meteostat les plus proches).

**Années :** si non fournies → **année en cours**. Les mois manquants de l'année en cours
sont complétés par le **même mois de l'année précédente** (jamais d'imputation à 0).


In [9]:
from pathlib import Path
import sys
from datetime import datetime

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "MeteoPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
ROD_OUTPUT = PREPARE / "RodPrep" / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

print("Année en cours :", datetime.utcnow().year)


Année en cours : 2026


## 1. Entrée — identité hôtel depuis RodPrep

Champs retenus : `hotel_code`, `hotel_name`, `hotel_brand`, `hotel_city`, `hotel_lat`, `hotel_lon`.

On rafraîchit l'entrée depuis RodPrep. Par défaut les années cibles = année en cours
(passer `target_years=(2024, 2025, …)` pour l'historique ventes).


In [11]:
from meteo_prep.prep import MeteoPrep, HOTEL_IDENTITY_COLS, READABLE_WEATHER, default_target_years

# Années non fournies → année en cours. Pour jointure ventes historiques, ex. :
# prep = MeteoPrep(INPUT_DIR, OUTPUT_DIR, target_years=(2023, 2024, 2025, 2026))
prep = MeteoPrep(INPUT_DIR, OUTPUT_DIR)

if not (ROD_OUTPUT / "hotel_lookup.parquet").exists():
    raise FileNotFoundError("Exécuter d'abord RodPrep/Explore/explore.ipynb")

hotels_path = prep.fill_input_from_rod(ROD_OUTPUT)
print("Entrée créée depuis RodPrep :", hotels_path)
print("Années cibles :", prep.target_years, "(défaut =", default_target_years(), ")")

hotels = prep.load_input()
print(f"Hôtels : {len(hotels)}")
geo_ok = hotels["hotel_lat"].notna() & hotels["hotel_lon"].notna()
print(f"Avec lat/lon : {geo_ok.sum()} / {len(hotels)}")
hotels[[c for c in HOTEL_IDENTITY_COLS if c in hotels.columns]]


ImportError: cannot import name 'default_target_years' from 'meteo_prep.prep' (/media/laghmari/ssd-data/dev/hotels/prepare/MeteoPrep/Src/meteo_prep/prep.py)

## 2. Météo au point hôtel (`hotel_lat`, `hotel_lon`)

`weather_for_hotel` interroge Meteostat au point `(lat, lon)` sur la fenêtre
années cibles + année précédente (pour l'imputation). Résultat indexé par `(annee, mois)`.


In [12]:
enrich_summary = []
for _, hotel in hotels.iterrows():
    info = prep.weather_for_hotel(hotel)
    by_ym = info.get("weather_by_year_month") or {}
    years = sorted({y for (y, _m) in by_ym.keys()}) if by_ym else []
    enrich_summary.append({
        "hotel_code": info["hotel_code"],
        "hotel_name": info["hotel_name"],
        "hotel_lat": info["hotel_lat"],
        "hotel_lon": info["hotel_lon"],
        "source": info["source"],
        "nb_mois_annee": len(by_ym),
        "annees": ",".join(str(y) for y in years),
        "nb_cles_meteo": info["nb_cles_meteo"],
        "warnings": "; ".join(info["warnings"]) if info["warnings"] else "",
    })

enrich_df = pd.DataFrame(enrich_summary)
print(f"Enrichissements : {len(enrich_df)} hôtels")
enrich_df


Enrichissements : 7 hôtels


,hotel_code,hotel_name,hotel_lat,hotel_lon,source,nb_mois_annee,annees,nb_cles_meteo,warnings
0,H2075,Ibis budget Nice Californie,43.689186,7.240512,hotel_coords,0,,288,
1,HB6A3,Ibis budget Strasbourg Centre République,48.591522,7.754599,hotel_coords,0,,288,
2,H0815,Ibis Styles Roissy CDG,49.006733,2.519843,hotel_coords,0,,252,
3,H6188,Mercure Paris Boulogne,48.833827,2.256274,hotel_coords,0,,288,
4,H0373,Mercure Paris Montmartre Sacré-Cœur,48.885048,2.329923,hotel_coords,0,,288,
5,HB5I0,Novotel Megève Mont-Blanc,45.859165,6.619055,hotel_coords,0,,252,
6,H3546,Novotel Paris Centre Tour Eiffel,48.849778,2.282836,hotel_coords,0,,288,


## 3. Aperçu — premier hôtel avec coordonnées


In [13]:
sample = hotels[hotels["hotel_lat"].notna() & hotels["hotel_lon"].notna()]
if sample.empty:
    sample = hotels
sample_hotel = sample.iloc[0]

info = prep.weather_for_hotel(sample_hotel)
by_ym = info.get("weather_by_year_month") or {}

print(
    f"Hôtel {info['hotel_code']} @ ({info['hotel_lat']}, {info['hotel_lon']}) "
    f"— {len(by_ym)} mois×années (source={info['source']})"
)

preview_rows = []
for (year, month), metrics in sorted(by_ym.items()):
    row = {"annee": year, "mois": month}
    for k in sorted(metrics)[:3]:
        row[k] = metrics[k]
    preview_rows.append(row)
pd.DataFrame(preview_rows).head(18)


Hôtel H2075 @ (43.689186, 7.240512) — 0 mois×années (source=hotel_coords)


""


## 4. Renommage lisible

Métriques déjà en `meteo_{libelle}_{stat}` (mean / min / max). Mapping brut → lisible :


In [14]:
print("Mapping métriques :", READABLE_WEATHER)
# Compat profil mensuel (année en cours si année absente des clés aplaties)
monthly_readable = prep._readable_monthly(by_ym)
readable_rows = []
for month, metrics in sorted(monthly_readable.items()):
    for col, val in sorted(metrics.items()):
        readable_rows.append({"mois": month, "colonne": col, "valeur": val})
readable_preview = pd.DataFrame(readable_rows)
print(f"Colonnes lisibles (année préférée) : {readable_preview['colonne'].nunique() if not readable_preview.empty else 0}")
readable_preview.head(18)


Mapping métriques : {'temp': 'temperature_c', 'dwpt': 'point_rosee_c', 'rhum': 'humidite_pct', 'prcp': 'precipitations_mm', 'snow': 'neige_mm', 'wspd': 'vent_kmh', 'pres': 'pression_hpa', 'tsun': 'ensoleillement_min'}
Colonnes lisibles (année préférée) : 0


""


## 5. Grille `hotel_code × annee × mois`

Une ligne par combinaison pour les années cibles **et** l'année précédente (source d'imputation).


In [15]:
rows = []
for _, hotel in hotels.iterrows():
    rows.extend(prep._rows_for_hotel(hotel))

expanded = pd.DataFrame(rows)
print(f"Grille brute : {expanded.shape[0]} lignes × {expanded.shape[1]} colonnes")
print(f"Hôtels : {expanded['hotel_code'].nunique()} — années : {sorted(expanded['annee'].unique())}")
expanded.sort_values(["hotel_code", "annee", "mois"]).head(12)


Grille brute : 168 lignes × 28 colonnes
Hôtels : 7 — années : [2024, 2025]


,hotel_code,hotel_name,annee,mois,meteo_temperature_c_mean,meteo_temperature_c_min,meteo_temperature_c_max,meteo_point_rosee_c_mean,meteo_point_rosee_c_min,meteo_point_rosee_c_max,meteo_humidite_pct_mean,meteo_humidite_pct_min,meteo_humidite_pct_max,meteo_precipitations_mm_mean,meteo_precipitations_mm_min,meteo_precipitations_mm_max,meteo_neige_mm_mean,meteo_neige_mm_min,meteo_neige_mm_max,meteo_vent_kmh_mean,meteo_vent_kmh_min,meteo_vent_kmh_max,meteo_pression_hpa_mean,meteo_pression_hpa_min,meteo_pression_hpa_max,meteo_ensoleillement_min_mean,meteo_ensoleillement_min_min,meteo_ensoleillement_min_max
96,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,1,4.919892,-5.3,12.6,2.820565,-6.5,8.9,86.686828,59.0,100.0,0.063710,0.0,3.2,0.160221,0.0,7.0,13.743011,1.1,39.2,1006.609409,988.9,1024.6,3.985215,0.0,51.0
97,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,2,8.523810,-1.3,18.4,6.003571,-4.3,12.4,84.879464,51.0,100.0,0.154167,0.0,2.4,0.000000,0.0,0.0,14.770536,1.4,30.6,1005.709077,979.1,1026.1,5.622024,0.0,60.0
98,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,3,9.261290,0.5,18.3,4.848790,-3.6,12.0,75.479839,36.0,100.0,0.055914,0.0,3.7,0.000000,0.0,0.0,11.252688,1.4,34.2,1019.915323,1003.8,1032.6,15.435484,0.0,60.0
99,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,4,13.293611,3.6,25.1,4.753472,-5.3,12.9,60.830556,14.0,99.0,0.011528,0.0,2.3,0.000000,0.0,0.0,11.123472,0.4,25.2,1020.388750,1007.4,1028.9,20.256944,0.0,60.0
100,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,5,16.671102,4.8,32.7,10.703629,-0.4,19.8,71.372312,23.0,99.0,0.140457,0.0,4.7,0.000000,0.0,0.0,9.567339,1.4,27.4,1016.911694,999.6,1030.3,17.143817,0.0,60.0
101,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,6,22.208392,9.6,39.6,12.501958,4.4,20.9,58.175524,16.0,95.0,0.056154,0.0,8.1,0.000000,0.0,0.0,11.086853,1.8,29.5,1018.080909,1004.1,1029.0,21.662937,0.0,60.0
102,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,7,20.669363,11.2,38.1,12.919629,2.4,19.4,65.184350,19.0,99.0,0.179841,0.0,12.3,0.000000,0.0,0.0,11.097215,1.4,24.5,1016.233289,1002.0,1029.2,18.298408,0.0,60.0
103,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,8,21.144086,12.3,36.1,12.172984,4.7,19.8,60.079301,22.0,97.0,0.049731,0.0,4.8,0.000000,0.0,0.0,10.214516,1.1,23.8,1016.618414,998.5,1027.9,21.108871,0.0,60.0
104,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,9,15.730417,7.7,28.8,11.139306,4.7,18.9,75.837500,36.0,100.0,0.076250,0.0,3.0,0.000000,0.0,0.0,12.468889,2.2,32.8,1017.319028,1004.9,1027.2,11.644444,0.0,60.0
105,H0373,Mercure Paris Montmartre Sacré-Cœur,2024,10,12.450941,6.3,18.2,9.042876,2.1,16.4,80.641129,46.0,100.0,0.047312,0.0,3.4,0.000000,0.0,0.0,13.541398,1.1,32.4,1016.667473,984.9,1033.6,9.763441,0.0,59.0


## 6. Imputation — mois manquants ← année précédente

Pour chaque `(hotel_code, annee, mois)` et chaque colonne `meteo_*` :
1. si la valeur est manquante → prendre le **même mois de l'année N-1** (puis N-2, …) ;
2. **jamais** d'imputation à `0`.

Les mois futurs de l'année en cours sont ainsi complétés par l'année dernière.


In [ ]:
meteo_cols = [c for c in expanded.columns if c.startswith("meteo_")]
missing_before = int(expanded[meteo_cols].isna().sum().sum()) if meteo_cols else 0

imputed = prep._impute_missing(expanded)
# Sortie finale : années cibles uniquement
imputed_targets = imputed[imputed["annee"].isin(prep.target_years)].copy()
missing_after = int(imputed_targets[meteo_cols].isna().sum().sum()) if meteo_cols else 0

print(f"NaN meteo avant imputation : {missing_before}")
print(f"NaN meteo après imputation (années cibles) : {missing_after}")
print("(les NaN restants = aucune valeur disponible sur les années antérieures non plus)")
imputed_targets.sort_values(["hotel_code", "annee", "mois"]).head(12)


## 7. Persistance `Output/`

`prep.run()` : entrée → météo lat/lon (année×mois) → imputation N←N-1 → filtre années cibles → fichiers.


In [ ]:
meteo_final = prep.run()
print(f"meteo_monthly : {meteo_final.shape}")
print(f"Années : {sorted(meteo_final['annee'].unique()) if not meteo_final.empty else []}")
print(f"Hôtels : {sorted(meteo_final['hotel_code'].dropna().unique())}")
display_cols = ["hotel_code", "hotel_name", "annee", "mois"] + [
    c for c in meteo_final.columns if c.startswith("meteo_temperature")
]
meteo_final[display_cols].head(12)

print("\nFichiers produits :")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" ", path.name, f"({path.stat().st_size} octets)")
